# Recovery, Revocation, Rollback, and Resilience

Recover a synthetic MCP service from a verified containment handoff to independently proven safe service. The central rule is **rollback is a strategy, not proof; target state and effects decide whether recovery succeeded**.

## Safety and scope

Everything is deterministic, synthetic, credential-free, and offline. No production registry, issuer, cluster, artifact repository, or customer system is contacted. The notebook imports canonical lab.py.

## Objectives

You will verify the handoff, converge distributed revocation, validate Ed25519 provenance and compatibility, choose a recovery strategy, reconcile ambiguous staging and promotion, test security and useful service, authorize public return separately, and reopen on re-entry.

In [ ]:
from dataclasses import replace
from datetime import timedelta
from pathlib import Path
import runpy

lab = runpy.run_path(Path("lab.py"))
(lab["PROTOCOL_VERSION"], lab["RECOVERY_POLICY_VERSION"])

## 1. Verify the containment handoff

Recovery inherits facts, evidence digests, affected authority, unresolved items, and allowed actions. It must verify them before creating a case.

In [ ]:
handoff = lab["create_handoff"]()
assert lab["verify_handoff"](handoff, now=lab["NOW"])
{
    "incident": handoff.incident_id,
    "control_revision": handoff.control_revision,
    "affected_hosts": handoff.affected_hosts,
    "unresolved": handoff.unresolved_items,
}

In [ ]:
tampered_handoff = replace(handoff, tenant_id="beta")
assert not lab["verify_handoff"](tampered_handoff, now=lab["NOW"])
lab["verify_handoff"](tampered_handoff, now=lab["NOW"])

## 2. Create an application-owned recovery case

Tenant and role checks precede persistence. A model hypothesis remains a cited proposal and never becomes fact or authority.

In [ ]:
repository = lab["RecoveryRepository"]()
manager = lab["RecoveryManager"](repository)
case = manager.accept_handoff(
    handoff,
    actor_id="coordinator-1",
    actor_roles=frozenset({"recovery_coordinator"}),
    allowed_tenants=frozenset({"acme"}),
    now=lab["NOW"] + timedelta(seconds=1),
)
(case.recovery_id, case.state.value, case.version)

In [ ]:
case = manager.add_model_hypothesis(
    case,
    proposed_by="assistant-model",
    text="rollback may be faster than a forward fix",
    evidence_ids=(handoff.evidence_package_digest,),
)
assert case.hypotheses[-1].accepted_as_fact is False
case.hypotheses[-1]

## 3. Revoke every authority path

The exact plan covers the registry, host caches, sessions, credentials, durable tasks, and egress. Every distributed control point must observe the new epoch.

In [ ]:
authority = lab["AuthorityPlane"](epoch=handoff.control_revision)
revocation_plan = lab["make_revocation_plan"](
    handoff,
    current_epoch=authority.epoch,
    now=lab["NOW"] + timedelta(seconds=2),
)
[(item.kind.value, item.target, item.secondary)
 for item in revocation_plan.actions]

In [ ]:
revocation_receipt = lab["approve_revocation"](
    revocation_plan,
    approver_id="commander-1",
    approver_role="incident_commander",
    now=lab["NOW"] + timedelta(seconds=2),
)
(revocation_receipt.receipt_id, revocation_receipt.proposal_digest)

### Inject a host-cache outage

A known unavailable control is partial, not successful. The same authorized plan can apply its missing idempotent action when the control recovers.

In [ ]:
revoker = lab["RevocationCoordinator"](authority)
partial = revoker.execute(
    "revoke-operation-1",
    revocation_plan,
    revocation_receipt,
    case,
    now=lab["NOW"] + timedelta(seconds=3),
    unavailable_kinds=frozenset({lab["RevocationKind"].INVALIDATE_CACHE}),
)
assert partial.status == lab["OperationStatus"].PARTIAL
(partial.status.value, partial.reason_code, dict(revoker.authority.observed_epochs))

In [ ]:
revocation_result = revoker.retry_missing(
    "revoke-operation-1",
    now=lab["NOW"] + timedelta(seconds=4),
)
assert revocation_result.status == lab["OperationStatus"].RECONCILED
(revocation_result.status.value, dict(revoker.authority.observed_epochs))

In [ ]:
revocation_proof = lab["prove_revocation"](
    revocation_plan,
    revoker.authority,
    handoff,
    (),
    proof_time=lab["NOW"] + timedelta(seconds=4),
)
assert revocation_proof.complete
revocation_proof

In [ ]:
late_effect = lab["EffectEvent"](
    "event-late",
    "acme",
    lab["NOW"] + timedelta(seconds=5),
    handoff.affected_workloads[0],
    handoff.affected_credentials[0],
    handoff.affected_sessions[0],
    handoff.affected_tasks[0],
    handoff.affected_destinations[0],
    lab["EffectOutcome"].COMPLETED,
    "protected.resource.audit",
)
failed_revocation_proof = lab["prove_revocation"](
    revocation_plan,
    revoker.authority,
    handoff,
    (late_effect,),
    proof_time=lab["NOW"] + timedelta(seconds=6),
)
assert not failed_revocation_proof.complete
failed_revocation_proof

In [ ]:
case = manager.mark_revocation_proven(
    case,
    revocation_proof,
    now=lab["NOW"] + timedelta(seconds=4),
    actor_roles=frozenset({"recovery_coordinator"}),
)
(case.state.value, case.version)

## 4. Verify the recovery candidate

The verifier checks exact bytes, immutable reference, Ed25519 provenance, trusted builder and source, build expectations, SBOM, vulnerabilities, incident remediation, schema compatibility, and time.

In [ ]:
candidate = lab["build_candidate"](now=lab["NOW"])
verifier = lab["CandidateVerifier"]()
candidate_verified, candidate_reasons = verifier.verify(
    candidate,
    required_remediation_ids=case.required_remediation_ids,
    current_schema_version="customer-schema-v4",
    now=lab["NOW"],
)
assert candidate_verified
(candidate.artifact_digest, candidate.immutable_reference, candidate_reasons)

In [ ]:
tampered_candidate = replace(candidate, artifact_bytes=b"tampered")
tampered_verified, tampered_reasons = verifier.verify(
    tampered_candidate,
    required_remediation_ids=case.required_remediation_ids,
    current_schema_version="customer-schema-v4",
    now=lab["NOW"],
)
assert not tampered_verified
tampered_reasons

## 5. Choose rollback, forward fix, or rebuild

Strategy depends on root cause, compatibility, data, and environment trust. A previous revision is not automatically known good.

In [ ]:
rollback = lab["select_strategy"](
    candidate,
    previous_digest=candidate.artifact_digest,
    previous_is_vulnerable=False,
    rebuild_required=False,
)
forward = lab["select_strategy"](
    candidate,
    previous_digest=lab["digest"](b"vulnerable-old"),
    previous_is_vulnerable=True,
    rebuild_required=False,
)
rebuild = lab["select_strategy"](
    candidate,
    previous_digest=candidate.artifact_digest,
    previous_is_vulnerable=False,
    rebuild_required=True,
)
(rollback.strategy.value, forward.strategy.value, rebuild.strategy.value)

## 6. Bind one exact staging plan

The plan includes strategy, candidate evidence, configuration, policy, capability contract, canary size, objectives, policy, case version, and expiry.

In [ ]:
decision = forward
recovery_plan = lab["make_recovery_plan"](
    case,
    candidate,
    decision,
    verifier,
    revocation_proof,
    current_schema_version="customer-schema-v4",
    now=lab["NOW"] + timedelta(seconds=5),
)
(recovery_plan.plan_id, recovery_plan.strategy.value,
 recovery_plan.canary_percent, recovery_plan.objectives)

In [ ]:
recovery_receipt = lab["approve_recovery"](
    recovery_plan,
    approver_id="commander-1",
    approver_role="recovery_commander",
    now=lab["NOW"] + timedelta(seconds=5),
)
(recovery_receipt.receipt_id, recovery_receipt.purpose,
 recovery_receipt.expires_at)

## 7. Reconcile an ambiguous stage

The deployment commits, but the simulated response is lost. A restarted executor reuses the operation ID and durable request digest, then confirms observed target state without creating another revision.

In [ ]:
executor = lab["RecoveryExecutor"]()
first_stage = executor.stage(
    "stage-operation-1",
    recovery_plan,
    recovery_receipt,
    candidate,
    case,
    now=lab["NOW"] + timedelta(seconds=6),
    simulate_lost_response=True,
)
assert first_stage.status == lab["OperationStatus"].UNKNOWN
(first_stage.status.value, executor.deployment.revision)

In [ ]:
restarted = lab["RecoveryExecutor"](executor.deployment, executor.journal)
staged = restarted.stage(
    "stage-operation-1",
    recovery_plan,
    recovery_receipt,
    candidate,
    case,
    now=lab["NOW"] + timedelta(seconds=7),
)
assert staged.status == lab["OperationStatus"].RECONCILED
assert restarted.deployment.revision == 1
(staged.status.value, staged.reason_code, restarted.deployment)

In [ ]:
case = manager.mark_staged(case, recovery_plan, staged)
(case.state.value, case.candidate_digest)

## 8. Validate security and useful service

A private canary must match every approved digest, pass labelled security regressions and valid tasks, complete no forbidden effects, show no compromise indicators, and use independent sources.

In [ ]:
observations = lab["build_valid_observations"](recovery_plan)
observations

In [ ]:
weak_observations = replace(observations, safe_tasks_succeeded=18)
weak_evidence = lab["validate_recovery"](
    recovery_plan,
    restarted.deployment,
    weak_observations,
    now=lab["NOW"] + timedelta(seconds=8),
)
assert not weak_evidence.complete
dict(weak_evidence.checks)

In [ ]:
evidence = lab["validate_recovery"](
    recovery_plan,
    restarted.deployment,
    observations,
    now=lab["NOW"] + timedelta(seconds=8),
)
assert evidence.complete and lab["verify_recovery_evidence"](evidence)
(evidence.evidence_id, evidence.evidence_digest, dict(evidence.checks))

In [ ]:
case = manager.mark_validated(case, evidence)
(case.state.value, case.validated_evidence_digest)

## 9. Authorize public return separately

A service owner signs a fresh single-use receipt over both the exact recovery plan and the signed validation evidence.

In [ ]:
return_receipt = lab["approve_return_to_service"](
    recovery_plan,
    evidence,
    approver_id="service-owner-1",
    approver_role="service_owner",
    now=lab["NOW"] + timedelta(seconds=9),
)
(return_receipt.receipt_id, return_receipt.purpose,
 return_receipt.proposal_digest)

In [ ]:
first_return = restarted.promote(
    "return-operation-1",
    recovery_plan,
    return_receipt,
    evidence,
    case,
    now=lab["NOW"] + timedelta(seconds=10),
    simulate_lost_response=True,
)
assert first_return.status == lab["OperationStatus"].UNKNOWN
(first_return.status.value, restarted.deployment.revision)

In [ ]:
return_restarted = lab["RecoveryExecutor"](
    restarted.deployment, restarted.journal
)
returned = return_restarted.promote(
    "return-operation-1",
    recovery_plan,
    return_receipt,
    evidence,
    case,
    now=lab["NOW"] + timedelta(seconds=11),
)
assert returned.status == lab["OperationStatus"].RECONCILED
(returned.status.value, return_restarted.deployment)

## 10. Prove safe service and reopen on re-entry

Public state, validation evidence, and independent post-return effects must agree. A late completed exfiltration effect falsifies recovery.

In [ ]:
safe_proof = lab["prove_safe_service"](
    recovery_plan,
    return_restarted.deployment,
    evidence,
    (),
    returned_at=returned.occurred_at,
)
assert safe_proof.complete
safe_proof

In [ ]:
case = manager.mark_returned(
    case,
    safe_proof,
    returned,
    now=returned.occurred_at,
)
(case.state.value, case.returned_at)

In [ ]:
reentry = lab["EffectEvent"](
    "event-reentry",
    "acme",
    returned.occurred_at + timedelta(seconds=1),
    "spiffe://acme/mcp/support-v3",
    "credential-support-v3",
    "session-recovered",
    "task-recovered",
    "exfil.example",
    lab["EffectOutcome"].COMPLETED,
    "protected.resource.audit",
)
failed_safe_proof = lab["prove_safe_service"](
    recovery_plan,
    return_restarted.deployment,
    evidence,
    (reentry,),
    returned_at=returned.occurred_at,
)
assert not failed_safe_proof.complete
failed_safe_proof

In [ ]:
reopened = manager.reopen(
    case,
    reason="independent post-recovery audit recorded forbidden egress",
    actor_roles=frozenset({"incident_responder"}),
)
assert reopened.state == lab["RecoveryState"].REOPENED
(reopened.state.value, reopened.reopen_reason)

## 11. Evaluate recovery and resilience

The metrics use labelled populations and exact denominators. Time to first process start is not time to safe service.

In [ ]:
demo = lab["run_demo"]()
demo["metrics"]

In [ ]:
demo["otel"]

## Exercises

1. Model a self-contained access-token propagation window and measure last accepted effect.
2. Make the previous artifact incompatible with the current schema and require forward recovery.
3. Treat the builder as compromised and rotate to an independent recovery trust root.
4. Design a degraded read-only MCP capability set and verify discovery, policy, and resource enforcement.
5. Partition one host at the old revocation epoch; explain why an average epoch is unsafe.
6. Redeliver a cancelled task after restart and prevent a duplicate protected-resource effect.
7. Add a data RPO and restore-point integrity check.
8. Write an approved game day with blast radius and abort criteria.

## Reflection

- Which authority can survive a registry revocation in your architecture?
- What exact evidence makes an old artifact known good?
- Which data or policy change makes rollback unsafe?
- Who may stage, validate, and authorize public return?
- Which independent outcome would falsify your recovery claim?
- What degraded service is safer than full outage for this mission?